In [ ]:
# Read-In AL0002_STORE_STATS_VW from Snowflake with needed columns and date formatting, sample cutting

import pandas as pd
from snowflake.snowpark import Session
from snowflake.snowpark.functions import sum, col
import pyarrow as pa
import pyarrow.parquet as pq
import os
from dotenv import load_dotenv

load_dotenv()

session = Session.builder.configs(connection_parameters).create()

query = "SELECT CLNDR_DT, DDM_ID, DOOR_SWINGS, RAW_GROSS_ACTIVATIONS, MARKET FROM SBX.SBX_BOOSTCBM.AL0002_STORE_STATS_VW"
store_df = session.sql(query).to_pandas()
store_df['CLNDR_DT'] = pd.to_datetime(store_df['CLNDR_DT'])

In [ ]:
# --- Local Data Clean-Up
'''This code cleans and consolidates the local and indirect performance data out of 
the local store data. This block attempts to clean store identifiers by
1. Fixing any potential data entry errors
2. Overwriting small moves (within a shopping center or down the street)
3. Looking for store IDs which changed address within a market and overwriting
4. Leaves 2 duplicated ID-Address pairs where the market changes.'''

from thefuzz import fuzz

# Assume 'store_df' is your initial, raw DataFrame.
def find_and_propose_replacements(df, similarity_threshold=85):
    """Finds similar addresses for each ID and proposes a canonical replacement."""
    grouped_by_id = df.groupby('DDM_ID')['STORE_ADDRESS'].apply(list).reset_index()
    proposed_changes = []
    for _, row in grouped_by_id.iterrows():
        ddm_id, addresses = row['DDM_ID'], row['STORE_ADDRESS']
        if len(addresses) < 2: continue
        
        clusters = []
        for addr in addresses:
            found_cluster = False
            for cluster in clusters:
                if fuzz.ratio(addr, cluster[0]) > similarity_threshold:
                    cluster.append(addr)
                    found_cluster = True
                    break
            if not found_cluster: clusters.append([addr])
        
        for cluster in clusters:
            if len(cluster) > 1:
                canonical_address = max(cluster, key=len)
                for original_address in cluster:
                    if original_address != canonical_address:
                        proposed_changes.append({
                            'DDM_ID': ddm_id,
                            'Original_Address': original_address,
                            'Proposed_Canonical_Address': canonical_address
                        })
    return pd.DataFrame(proposed_changes)

# Identify problematic IDs before the first cleaning round
initial_problem_ids = store_df.groupby('DDM_ID')['STORE_ADDRESS'].nunique()
initial_problem_ids = initial_problem_ids[initial_problem_ids > 1].index
initial_problem_df = store_df[store_df['DDM_ID'].isin(initial_problem_ids)][['DDM_ID', 'STORE_ADDRESS']].drop_duplicates()

# Generate the fuzzy matching cleanup plan
print("--- Round 1: Applying Fuzzy Matching for Typos ---")
fuzzy_changes_df = find_and_propose_replacements(initial_problem_df)

# Apply the fuzzy match changes
if not fuzzy_changes_df.empty:
    fuzzy_map = fuzzy_changes_df.set_index(['DDM_ID', 'Original_Address'])['Proposed_Canonical_Address'].to_dict()
    def apply_fuzzy_correction(row):
        return fuzzy_map.get((row['DDM_ID'], row['STORE_ADDRESS']), row['STORE_ADDRESS'])
    store_df['STORE_ADDRESS'] = store_df.apply(apply_fuzzy_correction, axis=1)
    print(f"✅ Fuzzy matching applied. {len(fuzzy_changes_df)} potential changes were identified.")
else:
    print("✅ No fuzzy matches found to correct.")


# --- ROUND 2: CONSOLIDATING STORE MOVES WITHIN THE SAME MARKET ---

print("\n--- Round 2: Consolidating store moves within the same market ---")
# Find IDs that STILL have multiple addresses after round 1
remaining_counts = store_df.groupby('DDM_ID')['STORE_ADDRESS'].nunique()
remaining_problem_ids = remaining_counts[remaining_counts > 1].index

if len(remaining_problem_ids) == 0:
    print("✅ No further consolidation needed after Round 1.")
else:
    # Isolate the remaining problem data
    remaining_problems_df = store_df[store_df['DDM_ID'].isin(remaining_problem_ids)][['DDM_ID', 'STORE_ADDRESS', 'MARKET']].drop_duplicates()
    
    consolidation_map = {}
    
    # Group by ID to check the markets
    for ddm_id, group in remaining_problems_df.groupby('DDM_ID'):
        # Check if all addresses for this ID share ONE market
        if group['MARKET'].nunique() == 1:
            # It's a safe consolidation candidate
            addresses = sorted(group['STORE_ADDRESS'].unique())
            canonical_address = addresses[0] # Pick the first alphabetically
            
            # Map all other addresses for this ID to the canonical one
            for original_address in addresses[1:]:
                consolidation_map[(ddm_id, original_address)] = canonical_address

    # Apply the consolidation changes
    if consolidation_map:
        def apply_consolidation(row):
            return consolidation_map.get((row['DDM_ID'], row['STORE_ADDRESS']), row['STORE_ADDRESS'])
        
        store_df['STORE_ADDRESS'] = store_df.apply(apply_consolidation, axis=1)
        print(f"✅ Consolidated {len(consolidation_map)} addresses for stores that moved within the same market.")
    else:
        print("✅ No clear intra-market moves found to consolidate.")


# --- FINAL REPORT ---
print("\n--- Final Integrity Check ---")
final_check = store_df.groupby('DDM_ID')['STORE_ADDRESS'].nunique()
final_issues = final_check[final_check > 1].index

if len(final_issues) == 0:
    print("\n🎉 SUCCESS! All DDM_IDs now map to a single, unique STORE_ADDRESS.")
else:
    print(f"\nFINAL ISSUES REMAIN: {len(final_issues)} DDM_IDs still have multiple addresses.")
    print("These may be stores that moved across markets or other complex cases.")
    
    final_issues_df = store_df[store_df['DDM_ID'].isin(final_issues)][['DDM_ID', 'STORE_ADDRESS', 'MARKET']].drop_duplicates().sort_values(by=['DDM_ID', 'MARKET'])
    
    print("\nRemaining Issues:")
    print(final_issues_df)

In [ ]:
# Renaming Treatment Columns

rename_map = {
    'CLNDR_DT': 'Date',
    'RAW_GROSS_ACTIVATIONS': 'GA'
}

# Perform the in-place rename operation
store_df.rename(columns=rename_map, inplace=True)

# Now, 'store_df' has the new column names
print(store_df.columns)

In [ ]:
# --- Dropping Local NAs

treatment_cols = ['OOH_SPEND', 'RADIO_SPEND', 'DIGITAL_SPEND', 'SPONSORSHIP_SPEND']
outcome_cols   = ['DOOR_SWINGS', 'GA']

# 2. Drop rows with missing in either treatment OR outcome
local_df_drop_na = store_df.dropna(subset=treatment_cols + outcome_cols)


In [ ]:
# --- Environment Cleanup
local_df = store_df.copy()

vars_to_delete = ['addresses', 'all_vals', 'ax', 'bins', 'canonical_address', 'CENSUS_API_KEY', 'columns_to_select', 'consolidation_map', 'date_col', 'ddm_id',
                  'END_DATE', 'exploration_sample_df', 'fig', 'final_check', 'final_issues', 'final_issues_df', 'fuzzy_changes_df', 'fuzzy_map', 'group',
                  'helper_cols', 'initial_problem_df', 'initial_problem_ids', 'market_mapping', 'missing', 'msa_to_counties', 'new_session', 'numeric', 'numeric_bins',
                  'original_address', 'outcome_cols', 'store_df', 'population_df', 'remaining_counts', 'remaining_problem_ids', 'remaining_problems_df', 'rename_map', 'snowpark_df',
                  'source_table', 'sp', 'spend_cols', 'sql_query', 'START_DATE', 'today', 'treatment_cols', 'var_name', 'VINTAGE', 'xticklabels', 'xticks', 'years_to_get',
                  ]

for var_name in vars_to_delete:
    if var_name in globals():
        del globals()[var_name]

del(vars_to_delete)

In [ ]:
# --- Local Marketing Missing Check
original_min = local_df['DATE'].min().date()
original_max = local_df['DATE'].max().date()

# Get the date range of the cleaned DataFrame
cleaned_min = local_df_drop_na['DATE'].min().date()
cleaned_max = local_df_drop_na['DATE'].max().date()

print("--- Date Range Comparison ---")
print(f"Original Range:  {original_min} to {original_max}")
print(f"Cleaned Range:   {cleaned_min} to {cleaned_max}")

# Get the row counts
original_rows = len(local_df)
cleaned_rows = len(local_df_drop_na)
rows_dropped = original_rows - cleaned_rows

# Print the comparison
print("--- Row Count Comparison ---")
print(f"Original row count:    {original_rows:,}")
print(f"After dropping NA:     {cleaned_rows:,}")
print(f"Rows dropped:          {rows_dropped:,}")

In [ ]:
canonical = local_marketing_panel['Market'].unique().tolist()

In [ ]:
g = set(search_by_market['Market'].unique())
l = set(local_marketing_panel['Market'].unique())
p = set(market_population_panel['Market'].unique())

# Convert sets to sorted lists for readability
g_list = sorted(g)
l_list = sorted(l)
p_list = sorted(p)

# Find the max length
max_len = max(len(g_list), len(l_list), len(p_list))

# Pad lists with empty strings to match max_len
g_list += [''] * (max_len - len(g_list))
l_list += [''] * (max_len - len(l_list))
p_list += [''] * (max_len - len(p_list))

# Create DataFrame
sets_df = pd.DataFrame({
    'search_by_market': g_list,
    'local_marketing_panel': l_list,
    'market_population_panel': p_list
})

sets_df.head()


sets_df.to_csv("C:/Users/ben.pharris/Documents/GitHub/project-dev/FunctionalEstimation/data/raw/MarketMatch.csv")

In [ ]:
# --- Search / Google Spend by Market
# This is used to test the robustness of the instrument through correlation of actual spending vs population

search_by_market = pd.read_csv("C:/Users/ben.pharris/Documents/GitHub/project-dev/FunctionalEstimation/data/raw/Spend by DMA.csv")
search_by_market_lf_only = pd.read_csv("C:/Users/ben.pharris/Documents/GitHub/project-dev/FunctionalEstimation/data/raw/Spend by DMA_LF.csv")

search_by_market = search_by_market.rename(columns={'DMA Region (Matched)': 'Market'})
search_by_market_lf_only = search_by_market_lf_only.rename(columns={'DMA Region (Matched)': 'Market'})


In [ ]:
# --- Applying the Search and Population Mappings for Instrument Robustness ---
# This is where the mapping actually happens. 1 - Use the search map dictionary in the map function. 2 - Aggregate search spending to the market level. 3- Create search spend percentages by market.
# 4- Align OpenSignal markets with Boost markets, and aggregate over the period to compare with search spending. 5 - Merge the two datasets to compare search spending % vs. population % by market.

import pandas as pd

# --- Step 1: Align Search Spending to Master Markets ---

# Map the search markets to the master market definitions
search_by_market['Master_Market'] = search_by_market['Market'].map(search_to_master_map)

# Group by the new master market and sum the costs
search_agg = search_by_market.groupby('Master_Market')['Cost'].sum().reset_index()

# Calculate the percentage of total search spend for each master market
total_search_cost = search_agg['Cost'].sum()
search_agg['Search_Percent'] = (search_agg['Cost'] / total_search_cost) * 100

print("Aggregated Search Spending by Master Market:")
print(search_agg.head())
print("-" * 50)


# --- Step 2: Align Population Data to Master Markets ---

# To get a single population percentage per market, we can average the monthly data.
# This is a reasonable approach if population distribution is relatively stable.
market_population_avg = market_population_panel.groupby('Market')['Population Percent'].mean().reset_index()

# Map the population markets to the master market definitions
market_population_avg['Master_Market'] = market_population_avg['Market'].map(population_to_master_map)

# Check for unmapped markets, print mapped
print("Number of unmapped markets:", market_population_avg['Master_Market'].isna().sum())

# Print unmapped market rows for debugging
print("Unmapped market rows:")
print(market_population_avg[market_population_avg['Master_Market'].isna()])

# gives the correct total percentage for the larger master market areas.
population_agg = market_population_avg.groupby('Master_Market')['Population Percent'].sum().reset_index()
population_agg = population_agg.rename(columns={'Population Percent': 'Population_Percent'})


print("Aggregated Population Percentage by Master Market:")
print(population_agg.head())
print("-" * 50)


# --- Step 3: Merge for Final Comparison ---

# Merge the two aggregated dataframes on the master market definition
market_comparison_df = pd.merge(
    search_agg[['Master_Market', 'Search_Percent']],
    population_agg[['Master_Market', 'Population_Percent']],
    on='Master_Market',
    how='outer' # Use an outer join to see if any markets exist in one dataset but not the other
)

# zero spend or zero population in a given market.
market_comparison_df = market_comparison_df.fillna(0)

print("Final Comparison of Search Spend % vs. Population %:")
print(market_comparison_df.head())

print(len(market_comparison_df['Master_Market'].unique()))


In [ ]:
# Mapping Robustness Check
# Check if all the the markets in the data set created in the previous cell match up with the population mapping dictionary.
population_markets = market_population_panel['Market'].unique()

# Get the keys from your mapping dictionary
map_keys = population_to_master_map.keys()

# Find which markets are in your data but not in your map
unmapped_markets = set(population_markets) - set(map_keys)

if unmapped_markets:
    print("\nMarkets in the data but not in the mapping dictionary:")
    print(unmapped_markets)
else:
    print("\nAll markets are successfully mapped!")

In [ ]:
# Population vs. Search Spend Correlation Graph.
# market_comparison_df is the merged dataframe containing both search spend and population percentages for the boost markets. This is the robustness check dataframe.

df_filtered = market_comparison_df[~market_comparison_df['Master_Market'].isin(['National', 'Other'])]

# 1. Recalculate the correlation on the filtered data
filtered_corr = df_filtered['Search_Percent'].corr(df_filtered['Population_Percent'])
print(f"Correlation (excluding 'National' market): {filtered_corr:.2f}")


# 2. Re-run the scatter plot on the filtered data
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

plt.style.use('dark_background')
# Create a font object
from matplotlib import font_manager as fm
poiret = fm.FontProperties(fname="C:/Users/ben.pharris/Downloads/Poiret_One/Montserrat-SemiBold.ttf",size=None)
from matplotlib import font_manager
font_manager.fontManager.addfont("C:/Users/ben.pharris/Downloads/Poiret_One/Montserrat-SemiBold.ttf")
plt.rcParams['font.family'] = poiret.get_name()
del(poiret)

plt.figure(figsize=(12, 8))
plt.style.use('dark_background')
sns.scatterplot(data=df_filtered, x='Population_Percent', y='Search_Percent', s=50 )

# Create text objects for each point
texts = [
    plt.text(row['Population_Percent'], row['Search_Percent'], row['Master_Market'], 
             fontsize=9, ha='left', va='bottom')
    for _, row in df_filtered.iterrows()
]

# Adjust texts: move them further away and add connecting arrows
adjust_text(
    texts, 
    arrowprops=dict(arrowstyle='->', color='white'),
    force_text=0.5,
    force_points=0.2,
    expand_text=(1.25, 1.25),
    expand_points=(1.2, 1.2)
)

plt.title('Search Spend % vs. Population % (Excluding National Market)')
plt.xlabel('Population Percentage')
plt.ylabel('Search Spend Percentage')
plt.grid(True)
plt.show()

In [ ]:
# Env Cleanup

del search_by_market, search_by_market_lf_only, search_agg
del market_population_avg, population_agg, market_comparison_df, df_filtered

# Mapping dictionaries used for the robustness test
del search_to_master_map

In [ ]:
search_to_master_map = {
    # --- Northeast ---
    'New York NY': 'New York City',
    'Boston MA (Manchester NH)': 'Boston',
    'Philadelphia PA': 'Philadelphia Metro',
    'Washington DC (Hagerstown MD)': 'Washington DC',
    'Baltimore MD': 'Baltimore',
    'Providence RI-New Bedford MA': 'Providence',
    'Harrisburg-Lancaster-Lebanon-York PA': 'Central Pennsylvania',
    'Wilkes Barre-Scranton-Hazleton PA': 'Central Pennsylvania',
    'Johnstown-Altoona-State College PA': 'Central Pennsylvania',
    'Albany-Schenectady-Troy NY': 'Upstate NY East',
    'Syracuse NY': 'Upstate NY Central',
    'Rochester NY': 'Rochester',
    'Buffalo NY': 'Buffalo',
    'Burlington VT-Plattsburgh NY': 'VT / NH / ME',
    'Portland-Auburn ME': 'VT / NH / ME',
    'Bangor ME': 'VT / NH / ME',
    'Presque Isle ME': 'VT / NH / ME',
    'Binghamton NY': 'Upstate NY Central',
    'Elmira NY': 'Upstate NY Central',
    'Utica NY': 'Upstate NY Central',
    'Watertown NY': 'Upstate NY Central',
    'Salisbury MD': 'Delaware',
    'Pittsburgh PA': 'Pittsburgh',
    'Wheeling WV-Steubenville OH': 'Pittsburgh', # Geographically fits Pittsburgh's region
    'Youngstown OH': 'Western Pennsylvania',

    # --- Midwest ---
    'Chicago IL': 'Chicago',
    'Detroit MI': 'East Michigan',
    'Minneapolis-St. Paul MN': 'Minnesota',
    'Cleveland-Akron (Canton) OH': 'Cleveland',
    'St. Louis MO': 'Missouri',
    'Indianapolis IN': 'Indianapolis',
    'Cincinnati OH': 'Cincinnati',
    'Kansas City MO': 'Kansas',
    'Columbus OH': 'Columbus',
    'Milwaukee WI': 'Milwaukee',
    'Grand Rapids-Kalamazoo-Battle Creek MI': 'West Michigan',
    'Dayton OH': 'Cincinnati',
    'Toledo OH': 'Toledo',
    'Green Bay-Appleton WI': 'North Wisconsin',
    'Ft. Wayne IN': 'Ft. Wayne / South Bend',
    'South Bend-Elkhart IN': 'Ft. Wayne / South Bend',
    'Madison WI': 'Milwaukee',
    'Rockford IL': 'Chicago',
    'Champaign & Springfield-Decatur IL': 'Central Illinois',
    'Peoria-Bloomington IL': 'Central Illinois',
    'Quincy IL-Hannibal MO-Keokuk IA': 'Central Illinois',
    'Flint-Saginaw-Bay City MI': 'East Michigan',
    'Lansing MI': 'East Michigan',
    'Traverse City-Cadillac MI': 'West Michigan',
    'Marquette MI': 'North Wisconsin',
    'Duluth MN-Superior WI': 'Minnesota',
    'Rochester MN-Mason City IA-Austin MN': 'Minnesota',
    'Mankato MN': 'Minnesota',
    'Wausau-Rhinelander WI': 'North Wisconsin',
    'La Crosse-Eau Claire WI': 'North Wisconsin',
    'Springfield MO': 'Missouri',
    'Columbia-Jefferson City MO': 'Missouri',
    'Paducah KY-Cape Girardeau MO-Harrisburg IL': 'Missouri',
    'St. Joseph MO': 'Kansas',
    'Topeka KS': 'Kansas',
    'Evansville IN': 'West Kentucky',
    'Terre Haute IN': 'Indianapolis',
    'Lafayette IN': 'Indianapolis',
    'Zanesville OH': 'Columbus',
    'Lima OH': 'Toledo', # Closest major Ohio canonical market
    'Parkersburg WV': 'West Virginia',
    'Alpena MI': 'East Michigan',

    # --- South ---
    'Atlanta GA': 'Atlanta / Athens',
    'Houston TX': 'Houston',
    'Dallas-Ft. Worth TX': 'DFW',
    'Miami-Ft. Lauderdale FL': 'Miami / West Palm',
    'Tampa-St. Petersburg (Sarasota) FL': 'Tampa',
    'Orlando-Daytona Beach-Melbourne FL': 'Orlando',
    'Charlotte NC': 'Charlotte',
    'Raleigh-Durham (Fayetteville) NC': 'Raleigh / Durham',
    'Nashville TN': 'Nashville',
    'San Antonio TX': 'San Antonio',
    'New Orleans LA': 'New Orleans',
    'Memphis TN': 'Memphis',
    'Greensboro-High Point-Winston Salem NC': 'Winston / Salem',
    'Jacksonville FL': 'Jacksonville',
    'Richmond-Petersburg VA': 'Richmond',
    'Norfolk-Portsmouth-Newport News VA': 'Norfolk',
    'Oklahoma City OK': 'Oklahoma',
    'Austin TX': 'Austin',
    'West Palm Beach-Ft. Pierce FL': 'Miami / West Palm',
    'Knoxville TN': 'Nashville',
    'Tulsa OK': 'Oklahoma',
    'Ft. Myers-Naples FL': 'South West Florida',
    'Baton Rouge LA': 'Louisiana',
    'Chattanooga TN': 'Nashville',
    'Greenville-New Bern-Washington NC': 'South Carolina', # Greenville is a major SC city
    'Wilmington NC': 'Myrtle Beach', # Geographically adjacent
    'Roanoke-Lynchburg VA': 'Southern Virginia',
    'Savannah GA': 'GA/SC Coast',
    'Tallahassee FL-Thomasville GA': 'The Panhandle',
    'Augusta GA': 'Georgia',
    'Macon GA': 'Georgia',
    'Columbus GA': 'Georgia',
    'Albany GA': 'Georgia',
    'Tyler-Longview(Lufkin & Nacogdoches) TX': 'East Texas',
    'Beaumont-Port Arthur TX': 'East Texas',
    'Corpus Christi TX': 'South Texas',
    'Laredo TX': 'South Texas',
    'Victoria TX': 'South Texas',
    'Waco-Temple-Bryan TX': 'Austin',
    'Panama City FL': 'The Panhandle',
    'Charlottesville VA': 'Southern Virginia',
    'Harrisonburg VA': 'Southern Virginia',
    'Gainesville FL': 'Jacksonville',
    'Biloxi-Gulfport MS': 'Gulf Coast',
    'Hattiesburg-Laurel MS': 'Mississippi',
    'Meridian MS': 'Mississippi',
    'Lafayette LA': 'Louisiana',
    'Lake Charles LA': 'Gulf Coast',
    'Alexandria LA': 'Louisiana',
    'Bowling Green KY': 'Nashville',
    'Tri-Cities TN-VA': 'Nashville',
    'Jackson TN': 'Memphis',
    'Jonesboro AR': 'Arkansas',
    'Bluefield-Beckley-Oak Hill WV': 'West Virginia',
    'Charleston-Huntington WV': 'West Virginia',
    'Clarksburg-Weston WV': 'West Virginia',

    # --- West ---
    'Los Angeles CA': 'LA Metro',
    'San Francisco-Oakland-San Jose CA': 'SF Bay',
    'Seattle-Tacoma WA': 'West Washington',
    'Phoenix (Prescott) AZ': 'Phoenix',
    'Denver CO': 'Colorado',
    'Sacramento-Stockton-Modesto CA': 'Upper Central Valley',
    'Portland OR': 'Oregon / SW Washington',
    'San Diego CA': 'San Diego',
    'Salt Lake City UT': 'Utah',
    'Las Vegas NV': 'Las Vegas',
    'Fresno-Visalia CA': 'Lower Central Valley',
    'Tucson (Sierra Vista) AZ': 'Tucson / Yuma',
    'Spokane WA': 'Inland Northwest',
    'Boise ID': 'Idaho',
    'Reno NV': 'SF Bay',
    'Santa Barbara-Santa Maria-San Luis Obispo CA': 'North LA',
    'Bakersfield CA': 'Lower Central Valley',
    'Palm Springs CA': 'Riverside / San Bernardino',
    'Colorado Springs-Pueblo CO': 'Colorado',
    'Grand Junction-Montrose CO': 'Colorado',
    'Sherman TX-Ada OK': 'DFW',
    'El Paso (Las Cruces) TX': 'West Texas',
    'Amarillo TX': 'West Texas',
    'Lubbock TX': 'West Texas',
    'Odessa-Midland TX': 'West Texas',
    'Abilene-Sweetwater TX': 'West Texas',
    'San Angelo TX': 'West Texas',
    'Eugene OR': 'Oregon / SW Washington',
    'Medford-Klamath Falls OR': 'Oregon / SW Washington',
    'Bend OR': 'Oregon / SW Washington',
    'Idaho Falls-Pocatello(Jackson WY) ID': 'Idaho',
    'Twin Falls ID': 'Idaho',
    'Yakima-Pasco-Richland-Kennewick WA': 'Inland Northwest',
    'Chico-Redding CA': 'Upper Central Valley',
    'Monterey-Salinas CA': 'SF Bay',
    'Eureka CA': 'SF Bay',
    'Yuma AZ-El Centro CA': 'Tucson / Yuma',
}

In [ ]:
# --- Env Cleanup

del filtered_corr, map_keys, missing_local_data, population_markets, texts, total_search_cost, unmapped_markets

In [ ]:
# --- Store Stats Read-In
session = Session.builder.configs(connection_parameters).create()

query = "SELECT CLNDR_DT, DDM_ID, DOOR_SWINGS, RAW_GROSS_ACTIVATIONS, MARKET, STORE_ADDRESS FROM SBX.SBX_BOOSTCBM.AL0002_STORE_STATS_VW"
store_df = session.sql(query).to_pandas()
store_df['CLNDR_DT'] = pd.to_datetime(store_df['CLNDR_DT'])

In [ ]:
# --- Fuzzy Matching for Store Address Cleanup

def find_and_propose_replacements(df, similarity_threshold=85):
    """Finds similar addresses for each ID and proposes a canonical replacement."""
    grouped_by_id = df.groupby('DDM_ID')['STORE_ADDRESS'].apply(list).reset_index()
    proposed_changes = []
    for _, row in grouped_by_id.iterrows():
        ddm_id, addresses = row['DDM_ID'], row['STORE_ADDRESS']
        if len(addresses) < 2: continue
        
        clusters = []
        for addr in addresses:
            found_cluster = False
            for cluster in clusters:
                if fuzz.ratio(addr, cluster[0]) > similarity_threshold:
                    cluster.append(addr)
                    found_cluster = True
                    break
            if not found_cluster: clusters.append([addr])
        
        for cluster in clusters:
            if len(cluster) > 1:
                canonical_address = max(cluster, key=len)
                for original_address in cluster:
                    if original_address != canonical_address:
                        proposed_changes.append({
                            'DDM_ID': ddm_id,
                            'Original_Address': original_address,
                            'Proposed_Canonical_Address': canonical_address
                        })
    return pd.DataFrame(proposed_changes)

# Identify problematic IDs before the first cleaning round
initial_problem_ids = store_df.groupby('DDM_ID')['STORE_ADDRESS'].nunique()
initial_problem_ids = initial_problem_ids[initial_problem_ids > 1].index
initial_problem_df = store_df[store_df['DDM_ID'].isin(initial_problem_ids)][['DDM_ID', 'STORE_ADDRESS']].drop_duplicates()

# Generate the fuzzy matching cleanup plan
print("--- Round 1: Applying Fuzzy Matching for Typos ---")
fuzzy_changes_df = find_and_propose_replacements(initial_problem_df)

# Apply the fuzzy match changes
if not fuzzy_changes_df.empty:
    fuzzy_map = fuzzy_changes_df.set_index(['DDM_ID', 'Original_Address'])['Proposed_Canonical_Address'].to_dict()
    def apply_fuzzy_correction(row):
        return fuzzy_map.get((row['DDM_ID'], row['STORE_ADDRESS']), row['STORE_ADDRESS'])
    store_df['STORE_ADDRESS'] = store_df.apply(apply_fuzzy_correction, axis=1)
    print(f"✅ Fuzzy matching applied. {len(fuzzy_changes_df)} potential changes were identified.")
else:
    print("✅ No fuzzy matches found to correct.")


# --- ROUND 2: CONSOLIDATING STORE MOVES WITHIN THE SAME MARKET ---

print("\n--- Round 2: Consolidating store moves within the same market ---")
# Find IDs that STILL have multiple addresses after round 1
remaining_counts = store_df.groupby('DDM_ID')['STORE_ADDRESS'].nunique()
remaining_problem_ids = remaining_counts[remaining_counts > 1].index

if len(remaining_problem_ids) == 0:
    print("✅ No further consolidation needed after Round 1.")
else:
    # Isolate the remaining problem data
    remaining_problems_df = store_df[store_df['DDM_ID'].isin(remaining_problem_ids)][['DDM_ID', 'STORE_ADDRESS', 'MARKET']].drop_duplicates()
    
    consolidation_map = {}
    
    # Group by ID to check the markets
    for ddm_id, group in remaining_problems_df.groupby('DDM_ID'):
        # Check if all addresses for this ID share ONE market
        if group['MARKET'].nunique() == 1:
            # It's a safe consolidation candidate
            addresses = sorted(group['STORE_ADDRESS'].unique())
            canonical_address = addresses[0] # Pick the first alphabetically
            
            # Map all other addresses for this ID to the canonical one
            for original_address in addresses[1:]:
                consolidation_map[(ddm_id, original_address)] = canonical_address

    # Apply the consolidation changes
    if consolidation_map:
        def apply_consolidation(row):
            return consolidation_map.get((row['DDM_ID'], row['STORE_ADDRESS']), row['STORE_ADDRESS'])
        
        store_df['STORE_ADDRESS'] = store_df.apply(apply_consolidation, axis=1)
        print(f"✅ Consolidated {len(consolidation_map)} addresses for stores that moved within the same market.")
    else:
        print("✅ No clear intra-market moves found to consolidate.")


# --- FINAL REPORT ---
print("\n--- Final Integrity Check ---")
final_check = store_df.groupby('DDM_ID')['STORE_ADDRESS'].nunique()
final_issues = final_check[final_check > 1].index

if len(final_issues) == 0:
    print("\n🎉 SUCCESS! All DDM_IDs now map to a single, unique STORE_ADDRESS.")
else:
    print(f"\nFINAL ISSUES REMAIN: {len(final_issues)} DDM_IDs still have multiple addresses.")
    print("These may be stores that moved across markets or other complex cases.")
    
    final_issues_df = store_df[store_df['DDM_ID'].isin(final_issues)][['DDM_ID', 'STORE_ADDRESS', 'MARKET']].drop_duplicates().sort_values(by=['DDM_ID', 'MARKET'])
    
    print("\nRemaining Issues:")
    print(final_issues_df)

In [ ]:
# Renaming Store Columns

rename_map = {
    'CLNDR_DT': 'Date',
    'RAW_GROSS_ACTIVATIONS': 'GA',
    'DOOR_SWINGS': 'Store Traffic',
    'MARKET': 'Market',
    'STORE_ADDRESS': 'Store Address'
}

# Perform the in-place rename operation
store_df.rename(columns=rename_map, inplace=True)

# Now, 'store_df' has the new column names
print(store_df.columns)

In [ ]:
del addresses, canonical_address, consolidation_map, ddm_id, final_check, final_issues, final_issues_df, final_panel, fuzzy_changes_df, fuzzy_map, group, initial_problem_df, initial_problem_ids,
del market, marketing_markets, markets_in_mkt_only, markets_in_pop_only, original_address, remaining_counts, remaining_problem_ids, remaining_problems_df, population_markets